<a href="https://colab.research.google.com/github/catafest/colab_google/blob/master/catafest_076.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install astroquery gradio

This script creates an interactive web application using the Gradio library called "Astronomical Archive Scanner". Its purpose is to scan, download, and cache astronomical data for any specified celestial object (like the comet 2P/Encke) from 5 major astronomical databases and observatories.
Key Components
1. File Deduplication (checksum & save_if_new)

    checksum(path): Calculates a unique SHA-256 hash of a file's content.
    save_if_new(data, out): Prevents downloading identical files twice. It only saves the newly queried data if the target file does not exist locally or if its content has changed compared to the existing local version.

2. Querying the Archives

It queries five different scientific services using astroquery:

    Helioviewer (check_helioviewer): Downloads solar images matching the current UTC time from space missions like SDO (using AIA channels) and SOHO (using LASCO coronagraphs).
    SkyView (check_skyview): Accesses multi-wavelength sky surveys (like DSS, PanSTARRS, WISE, GALEX) to retrieve Fits files of the region around your target object.
    MAST (check_mast): Queries the Mikulski Archive for Space Telescopes (which hosts data from Hubble, Kepler, etc.) for any public observation products recorded near the object's coordinates.
    Gaia (check_gaia): Performs an asynchronous database query on the Gaia DR3 catalog to retrieve stellar sources within a small radius of the target coordinates and saves the results in a CSV table.
    MPC - Minor Planet Center (check_mpc): Retrieves orbital ephemerides (predicted positions over time) specifically for solar system bodies like comets and asteroids.

3. Orchestration (run_all)

    This function runs all five check functions in sequence, saves any newly retrieved files to organized local directories under ./data, and merges all success/error logs and file paths to return them to the user interface.

4. The GUI (demo)

    gr.Blocks(): Builds the visual user interface where you can type in the name of an object, click "Scan archives", view real-time log details, and view/download all gathered files directly from your browser.

In [ ]:
import os
import hashlib
from datetime import datetime
import gradio as gr

import astroquery
from astroquery.skyview import SkyView
from astroquery.mast import Observations
from astroquery.gaia import Gaia
from astroquery.mpc import MPC

def checksum(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        h.update(f.read())
    return h.hexdigest()

def save_if_new(data, out):
    if not os.path.exists(out):
        with open(out, "wb") as f:
            f.write(data)
        return True
    old = checksum(out)
    new = hashlib.sha256(data).hexdigest()
    if old != new:
        with open(out, "wb") as f:
            f.write(data)
        return True
    return False

def check_helioviewer(obj, outdir):
    # Helioviewer was deprecated and removed from astroquery. This is now a placeholder.
    return ["Helioviewer is deprecated in newer astroquery versions"], []

def check_skyview(obj, outdir):
    os.makedirs(outdir, exist_ok=True)
    surveys = ["DSS", "DSS2 Red", "DSS2 Blue", "PanSTARRS", "WISE 3", "GALEX"]
    logs = []
    files = []
    for s in surveys:
        try:
            imgs = SkyView.get_images(position=obj, survey=[s])
            if not imgs:
                logs.append(f"SkyView none: {s}")
                continue
            hdu = imgs[0][0]
            data = hdu.data
            out = os.path.join(outdir, f"{obj}_{s}.fits")
            if save_if_new(data.tobytes(), out):
                logs.append(f"SkyView new: {s}")
                files.append(out)
            else:
                logs.append(f"SkyView no new: {s}")
        except:
            logs.append(f"SkyView error: {s}")
    return logs, files

def check_mast(obj, outdir):
    os.makedirs(outdir, exist_ok=True)
    logs = []
    files = []
    try:
        obs = Observations.query_object(obj, radius="0.02 deg")
        if len(obs) == 0:
            logs.append("MAST none")
            return logs, files
        products = Observations.get_product_list(obs)
        dl = Observations.download_products(products, download_dir=outdir)
        logs.append("MAST downloaded")
        for f in dl:
            files.append(f["Local Path"])
    except:
        logs.append("MAST error")
    return logs, files

def check_gaia(obj, outdir):
    os.makedirs(outdir, exist_ok=True)
    logs = []
    files = []
    try:
        job = Gaia.launch_job_async(
            f"SELECT * FROM gaiadr3.gaia_source WHERE CONTAINS(POINT('ICRS',ra,dec),CIRCLE('ICRS',(SELECT ra FROM gaiadr3.gaia_source LIMIT 1),(SELECT dec FROM gaiadr3.gaia_source LIMIT 1),0.05))=1"
        )
        r = job.get_results()
        out = os.path.join(outdir, f"{obj}_gaia.csv")
        data = r.to_pandas().to_csv().encode()
        if save_if_new(data, out):
            logs.append("Gaia new")
            files.append(out)
        else:
            logs.append("Gaia no new")
    except:
        logs.append("Gaia error")
    return logs, files

def check_mpc(obj, outdir):
    os.makedirs(outdir, exist_ok=True)
    logs = []
    files = []
    try:
        eph = MPC.get_ephemeris(obj)
        out = os.path.join(outdir, f"{obj}_mpc.txt")
        data = eph.to_string().encode()
        if save_if_new(data, out):
            logs.append("MPC new")
            files.append(out)
        else:
            logs.append("MPC no new")
    except:
        logs.append("MPC error")
    return logs, files

def run_all(object_name):
    base = "./data"
    hv_logs, hv_files = check_helioviewer(object_name, os.path.join(base, "hv"))
    sky_logs, sky_files = check_skyview(object_name, os.path.join(base, "sky"))
    mast_logs, mast_files = check_mast(object_name, os.path.join(base, "mast"))
    gaia_logs, gaia_files = check_gaia(object_name, os.path.join(base, "gaia"))
    mpc_logs, mpc_files = check_mpc(object_name, os.path.join(base, "mpc"))

    logs = hv_logs + sky_logs + mast_logs + gaia_logs + mpc_logs
    files = hv_files + sky_files + mast_files + gaia_files + mpc_files

    return "\n".join(logs), files

with gr.Blocks() as demo:
    gr.Markdown("## Astronomical Archive Scanner")
    obj = gr.Textbox(label="Object name", value="2P/Encke")
    btn = gr.Button("Scan archives")
    out_logs = gr.Textbox(label="Logs")
    out_files = gr.File(label="Downloaded files", file_count="multiple")
    btn.click(run_all, obj, [out_logs, out_files])

demo.launch()

The current archive release 3.15 contains two breaking changes. Please see https://www.cosmos.esa.int/web/gaia/news#GACSrelease315 for details. In case of questions or feedback, please consult the Gaia helpdesk at https://www.cosmos.esa.int/web/gaia/gaia-helpdesk.
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bf3f0d9ab750341f3d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
